In [0]:
# Databricks notebook source
# ============================================================================
# 01. BRONZE LAYER - HIGH-THROUGHPUT INGESTION
# Objective: Ingest 20,000 raw multiline JSON CPQ payloads into Delta Lake.
# ============================================================================

from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType
from pyspark.sql.functions import explode, col

# # 1. DEFINE WIDGET (Input Parameter with default value)
# dbutils.widgets.text("batch_folder", "batch1_raw", "Batch Folder Name")

# 2. GET PARAMETER VALUE
batch_folder = dbutils.widgets.get("batch_folder")

# 3. DYNAMICALLY CONSTRUCT RAW PATH
RAW_PATH = f"/Volumes/cpq_dev_ws/default/manufacturing_pipeline_files/raw_cpq_data/{batch_folder}/"
BRONZE_TABLE = "default.bronze_cpq_quotes"

print(f"🚀 Processing Batch Folder: '{batch_folder}' from path: {RAW_PATH}")

# COMMAND ----------
# 4. Read Raw JSON
df_raw = spark.read \
    .option("multiline", "false") \
    .json(RAW_PATH)
print("Reading json file")

# 5. Check if the root level is an outer array or a flat list of JSON objects
if "value" in df_raw.columns:
    df_bronze = df_raw.select(explode("value").alias("quote")).select("quote.*")
else:
    df_bronze = df_raw

# 6. Add audit metadata columns
df_bronze = df_bronze.withColumn("ingestion_time", current_timestamp()) \
                      .withColumn("source_file", col("_metadata.file_path"))

# 7. Save to Bronze Delta Table
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRONZE_TABLE)
print(f" Successful Bronze Ingestion: {df_bronze.count():,} records stored in {BRONZE_TABLE}")